# YOLO11n-pose — landing base @ imgsz 1280 (Kaggle)

> **Project authored by Claude (Anthropic, Opus 4.8)** for the `a6_gaivotas_ccd2026` project.

Trains yolo11n-pose (center + 4 corners) at **imgsz 1280** on the Label Studio export.
Single-GPU (Kaggle API kernels give one P100). Attach dataset `yabrick/base-pouso-pose-ls91`,
GPU + Internet ON. Output: `best.pt` copied to `/kaggle/working/best_1280.pt`.

In [ ]:
!pip install -U ultralytics -q
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

import torch
_orig_load = torch.load
def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    if pickle_module is not None:
        return _orig_load(f, map_location=map_location, pickle_module=pickle_module, **kwargs)
    return _orig_load(f, map_location=map_location, **kwargs)
torch.load = _patched_load

import ultralytics
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

In [ ]:
from pathlib import Path
input_root = Path('/kaggle/input')
for p in sorted(input_root.rglob('*'))[:20]:
    print(p)

In [ ]:
# Convert Label Studio JSON -> YOLO Pose (center + 4 corners, fixed slot order).
import json, shutil, math, numpy as np
from pathlib import Path

ls_json = next(Path('/kaggle/input').rglob('*.json'))
print('Label Studio JSON:', ls_json)
with open(ls_json) as f:
    tasks = json.load(f)

images_root = Path('/kaggle/input')
out_images = Path('/kaggle/working/dataset_raw/images'); out_images.mkdir(parents=True, exist_ok=True)
out_labels = Path('/kaggle/working/dataset_raw/labels'); out_labels.mkdir(parents=True, exist_ok=True)

def ls_rect_to_yolo_aabb(rect):
    x_pct, y_pct = rect['x'], rect['y']
    w_pct, h_pct = rect['width'], rect['height']
    angle = np.radians(rect.get('rotation', 0))
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    corners = np.array([[0, 0], [w_pct, 0], [w_pct, h_pct], [0, h_pct]], dtype=float)
    rx = corners[:, 0] * cos_a - corners[:, 1] * sin_a + x_pct
    ry = corners[:, 0] * sin_a + corners[:, 1] * cos_a + y_pct
    x1, x2 = np.clip(rx.min(), 0, 100), np.clip(rx.max(), 0, 100)
    y1, y2 = np.clip(ry.min(), 0, 100), np.clip(ry.max(), 0, 100)
    return ((x1 + x2) / 2 / 100, (y1 + y2) / 2 / 100, (x2 - x1) / 100, (y2 - y1) / 100)

CANON = [math.atan2(-1, -1), math.atan2(-1, 1), math.atan2(1, 1), math.atan2(1, -1)]  # TL,TR,BR,BL
def ang_dist(a, b):
    d = abs(a - b) % (2 * math.pi)
    return min(d, 2 * math.pi - d)

def assign_corners(center, corners):
    cx, cy = center
    pairs = []
    for ci, (x, y) in enumerate(corners):
        a = math.atan2(y - cy, x - cx)
        for si, ref in enumerate(CANON):
            pairs.append((ang_dist(a, ref), ci, si))
    pairs.sort()
    slots = [None] * 4; used_c, used_s = set(), set()
    for _, ci, si in pairs:
        if ci in used_c or si in used_s:
            continue
        slots[si] = corners[ci]; used_c.add(ci); used_s.add(si)
    return [(s[0], s[1], 2) if s is not None else (0.0, 0.0, 0) for s in slots]

converted = skipped = 0
for task in tasks:
    img_name = Path(task.get('data', {}).get('image', '')).name
    src_img = next(images_root.rglob(img_name), None)
    if src_img is None:
        skipped += 1; continue
    annotations = task.get('annotations', [])
    if not annotations:
        skipped += 1; continue
    lines = []
    for ann in annotations:
        result = ann.get('result', [])
        rects = [r for r in result if r.get('type') == 'rectanglelabels']
        kc = [r for r in result if r.get('type') == 'keypointlabels' and r['value'].get('keypointlabels', [''])[0] == 'center']
        kk = [r for r in result if r.get('type') == 'keypointlabels' and r['value'].get('keypointlabels', [''])[0] == 'corner']
        if not rects or not kc:
            continue
        cx, cy, bw, bh = ls_rect_to_yolo_aabb(rects[0]['value'])
        c = kc[0]['value']; center = (c['x'] / 100, c['y'] / 100)
        corners = [(r['value']['x'] / 100, r['value']['y'] / 100) for r in kk]
        slots = assign_corners(center, corners)
        parts = [f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}", f"{center[0]:.6f} {center[1]:.6f} 2"]
        for x, y, v in slots:
            parts.append(f"{x:.6f} {y:.6f} {v}")
        lines.append(" ".join(parts))
    if not lines:
        skipped += 1; continue
    (out_labels / f'{Path(img_name).stem}.txt').write_text('\n'.join(lines))
    shutil.copy(src_img, out_images / img_name)
    converted += 1
print(f'Converted: {converted}, skipped: {skipped}')

In [ ]:
# Split 80/20
import random, shutil
from pathlib import Path
random.seed(42)
images = sorted(Path('/kaggle/working/dataset_raw/images').glob('*'))
random.shuffle(images)
split = int(len(images) * 0.8)
for name, imgs in [('train', images[:split]), ('val', images[split:])]:
    Path(f'/kaggle/working/dataset/images/{name}').mkdir(parents=True, exist_ok=True)
    Path(f'/kaggle/working/dataset/labels/{name}').mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy(img, f'/kaggle/working/dataset/images/{name}/{img.name}')
        lbl = Path('/kaggle/working/dataset_raw/labels') / (img.stem + '.txt')
        if lbl.exists():
            shutil.copy(lbl, f'/kaggle/working/dataset/labels/{name}/{lbl.name}')
print('train', split, '| val', len(images) - split)

In [ ]:
yaml_content = """path: /kaggle/working/dataset
train: images/train
val: images/val
nc: 1
names: ['base_pouso']
kpt_shape: [5, 3]
flip_idx: [0, 2, 1, 4, 3]
"""
open('/kaggle/working/data.yaml', 'w').write(yaml_content)
print(yaml_content)

In [ ]:
import os
os.chdir('/kaggle/working')
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=150,
    imgsz=1280,        # <- alta resolucao p/ cantos pequenos
    batch=8,           # <- 1280 pesa 4x mais que 640; batch pequeno p/ caber em 1 GPU
    device=0,          # <- Kaggle API kernel = 1 GPU (P100)
    project='base_pouso', name='run1280', patience=40,
    lr0=0.001, lrf=0.01, cos_lr=False,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.2,
    degrees=10, fliplr=0.4, flipud=0.2, scale=0.5, translate=0.0,
)
from pathlib import Path
RUN_DIR = Path(model.trainer.save_dir)
print('RUN_DIR:', RUN_DIR)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
df = pd.read_csv(RUN_DIR / 'results.csv'); df.columns = df.columns.str.strip()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(df['epoch'], df['metrics/mAP50(P)'], label='mAP50 (pose)')
ax[0].plot(df['epoch'], df['metrics/mAP50-95(P)'], label='mAP50-95 (pose)')
ax[0].set_title('mAP - Pose'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(df['epoch'], df['train/pose_loss'], label='train'); ax[1].plot(df['epoch'], df['val/pose_loss'], label='val')
ax[1].set_title('Pose loss'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()
print('Best mAP50(P):    %.4f' % df['metrics/mAP50(P)'].max())
print('Best mAP50-95(P): %.4f' % df['metrics/mAP50-95(P)'].max())

In [ ]:
import shutil
from pathlib import Path
best = next(Path('/kaggle/working').rglob('weights/best.pt'))
shutil.copy(best, '/kaggle/working/best_1280.pt')
print('best.pt:', best)
print('-> /kaggle/working/best_1280.pt (%.1f MB)' % (best.stat().st_size / 1e6))